## Configurações iniciais

### 2. Importações

In [ ]:
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from pathlib import Path

from src.reajustes import (
    RE_VOTO_DIRETO,
    detectar_reajuste_de_voto,
    exportar_reajustes_de_voto,
    tramitacao_dois_ambientes,
    verificar_sustentacao_oral,
    vincular_reajuste_a_inclusoes,
    vincular_sustentacao_a_inclusoes,
)


### 4. Importando os Datasets

In [ ]:
PROCESSED_PATH = Path('data/processed')
INTERIM_PATH = Path('data/interim')

df_andamentos = pd.read_parquet(
    INTERIM_PATH / 'dim_andamentos.parquet',
    engine='pyarrow'
)

print(f"Andamentos: {len(df_andamentos):,} linhas × {df_andamentos.shape[1]} colunas")
print(f"Colunas: {df_andamentos.columns.tolist()}")
display(df_andamentos.head())


In [ ]:
df_final = pd.read_parquet(
    PROCESSED_PATH / 'inclusoes_em_pauta.parquet',
    engine='pyarrow'
)

print(f"Inclusões em pauta: {len(df_final):,} linhas")
print(f"Colunas de df_final: {df_final.columns.tolist()}")
display(df_final.head())


## Detecção de reajuste de voto

### Termos que configuram reajustes

Cobre 100% das expressões da especificação:
- REAJUSTE     — reajuste, reajustado(s), reajustou, reajustaram, parcialmente reajustado, voto ora reajustado, aditado/ajustava
- ALTERAÇÃO    — alteração de/do voto, voto alterado(s), alteraram os/seus votos
- REFORMULAÇÃO — voto reformulado(s), após o voto reformulado

In [ ]:
# RE_VOTO_DIRETO e RE_MINISTRO agora vivem em src.reajustes (importadas acima)


### Casos de falsos positivos de reajuste de voto

O texto pode conter outros tipos de "reajuste". Ex: Reajuste salarial/de vencimentos (falso positivo) — "reajuste de seus vencimentos no percentual de 47,94%". Com isso em mente, pedi para o DeepSeek analisar todos os casos andamentos, onde foi identificados os seguintes casos:

| Padrão | Descrição | Exemplos de gatilhos textuais |
|--------|-----------|-------------------------------|
| **1. Reajuste salarial / de vencimentos** | O termo “reajuste” refere‑se a aumento de remuneração de servidores, magistrados, deputados etc., e não a mudança de voto. | “reajuste de vencimentos”, “reajuste de 11,98%”, “reajuste anual dos servidores”, “reajuste automático da remuneração”, “reajustes conferidos aos servidores”, “reajuste mantido”, “reajustes salariais”, “índices de reajuste”, “subsídio será reajustado”, “reajuste de 102% a servidores” |
| **2. Reajuste de preços / mensalidades** | “Reajuste” incide sobre valores de contratos, como mensalidades de planos de saúde. | “reajuste anual da mensalidade”, “reajuste de preço” |
| **3. Reformulação de sites / sistemas** | “Reformular” ou “reformulação” aplicado ao Portal da Transparência, não a voto. | “Portal da Transparência já foi reformulado”, “proposta de reformulação do Portal” |
| **4. Aditamento processual não relacionado a voto** | “Aditamento” aparece como peça processual genérica (ex.: aditamento da inicial) ou como documento dirigido ao relator, sem indicação de que se trata de complemento de voto. | “ADITAMENTO - AO MINISTRO RELATOR”, “converter o julgamento em diligência para aditamento da inicial” |
| **5. Citação de texto legal com o verbo “reajustar”** | O verbo aparece numa enumeração de competências legais (fixar, reajustar, revisar, homologar), fora do contexto de julgamento. | “expressões ‘fixar, reajustar, revisar, homologar’” |

Em resumo:
- Um reajuste de voto real sempre menciona "voto" perto da expressão:
  - "reajustou seu voto"
  - "voto ora reajustado"
  - "reajuste de voto do Relator"
  - "aditou seu voto"
  - "reformulou o voto"

- Já os falsos positivos ligam a expressão a outra coisa:

  - "reajuste de vencimentos" (não tem "voto" perto)
  - "reformulação do Portal" (não tem "voto" perto)
  - "aditamento da inicial" (não tem "voto" perto)

In [ ]:
def imprimir_barra_progresso(valor, max_valor, largura=30):
    if max_valor == 0: return ""
    blocos = int((valor / max_valor) * largura)
    return "█" * blocos

reaj = detectar_reajuste_de_voto(df_andamentos)

print("="*62)
print("DETECÇÃO DE REAJUSTE DE VOTO (2020–2025)")
print("="*62)
print(f"Total de andamentos com reajuste: {len(reaj):,}")
print(f"Incidentes distintos:             {reaj['incidente'].nunique():,}")

print("\nPor ano:")
contagem_ano = reaj['and_data_dt'].dt.year.value_counts().sort_index()
max_ano = contagem_ano.max() if not contagem_ano.empty else 0
for ano, qtd in contagem_ano.items():
    barra = imprimir_barra_progresso(qtd, max_ano)
    print(f"  {ano}  {qtd:>3}  {barra}")

print("\nPor classe:")
contagem_classe = reaj['classe'].value_counts()
total_reaj = len(reaj)
for classe, qtd in contagem_classe.items():
    pct = (qtd / total_reaj) * 100
    print(f"  {classe:<6} {qtd:>4} ({pct:>5.1f}%)")

# ── Ligação às inclusões em pauta ────────────────────────────────────────
df_final = vincular_reajuste_a_inclusoes(df_final, reaj)

print("\n" + "="*62)
print("LIGAÇÃO ÀS INCLUSÕES EM PAUTA")
print("="*62)
print(f"Inclusões com reajuste de voto:   {df_final['teve_reajuste'].sum():,}")
print(f"Processos distintos afetados:     {df_final[df_final['teve_reajuste']]['incidente'].nunique():,}")

print("\nPor ambiente:")
contagem_amb = df_final[df_final['teve_reajuste']]['ambiente'].value_counts()
total_inc_reaj = df_final['teve_reajuste'].sum()
for amb, qtd in contagem_amb.items():
    pct = (qtd / total_inc_reaj) * 100
    print(f"  {amb:<18} {qtd:>4} ({pct:>5.1f}%)")

print("\nPor ano:")
contagem_ano_f = df_final[df_final['teve_reajuste']]['ano'].value_counts().sort_index()
max_ano_f = contagem_ano_f.max() if not contagem_ano_f.empty else 0
for ano, qtd in contagem_ano_f.items():
    barra = imprimir_barra_progresso(qtd, max_ano_f)
    print(f"  {ano}  {qtd:>3}  {barra}")


### Validação manual da detecção

#### Todos os casos que configuram reajuste de voto

In [ ]:
print(f"Todos os {len(reaj):,} casos que configuram reajuste de voto:\n")
for _, row in reaj.iterrows():
    t = str(row['and_complemento'])
    m = RE_VOTO_DIRETO.search(t)   # regex final, a mesma usada na detecção
    if m:
        trecho = t[max(0, m.start()-45):m.end()+45]
        print(f"  ...{trecho}...")


### Salvamento

Exportação como planilha dos diversos casos que apresentam reajustes de votos

In [ ]:
reaj_export = exportar_reajustes_de_voto(df_final, reaj)

print(f"Total de andamentos de reajuste: {len(reaj_export):,}")
print(f"Processos distintos: {reaj_export['incidente'].nunique():,}")


In [ ]:
reaj_export.to_parquet(INTERIM_PATH / 'reajustes_de_voto.parquet', index=False)
print(f"Arquivo salvo: {INTERIM_PATH / 'reajustes_de_voto.parquet'}")


### Visualização

In [12]:
CORES_CLASSE = {
    'ADI':  '#2563eb',
    'ADPF': '#f59e0b',
    'ADC':  '#16a34a',
    'ADO':  '#ef4444',
}

CORES_REAJUSTE = {
    'Com reajuste de voto': '#dc2626',   # vermelho
    'Sem reajuste de voto': '#e5e7eb',   # cinza claro
}

In [13]:
def plotar_barras_reajuste(df_dados, col_x, col_grupo=None, titulo='',
                            label_y='Inclusões com reajuste', cores=None):
    """Barras (agrupadas ou simples) para contagens de reajuste."""
    fig = go.Figure()

    if col_grupo:
        grupos = sorted(df_dados[col_grupo].unique())
        for g in grupos:
            sub = df_dados[df_dados[col_grupo] == g]
            fig.add_trace(go.Bar(
                x=sub[col_x], y=sub['n'], name=str(g),
                marker_color=(cores.get(g) if cores else None),
                text=sub['n'], textposition='outside',
            ))
        fig.update_layout(barmode='group')
    else:
        fig.add_trace(go.Bar(
            x=df_dados[col_x], y=df_dados['n'],
            text=df_dados['n'], textposition='outside',
            marker_color='#dc2626',
        ))

    fig.update_layout(
        title=dict(text=titulo, x=0.5, xanchor='center', font=dict(size=14)),
        xaxis=dict(title='', dtick=1),
        yaxis=dict(title=label_y),
        margin=dict(t=60, b=40, l=50, r=30),
        height=440,
        legend=dict(orientation='h', yanchor='top', y=-0.12, xanchor='center', x=0.5),
    )
    fig.show()
    return fig

In [14]:
def plotar_pizza_reajuste(serie, titulo, cores=None):
    """Pizza simples para proporção com/sem reajuste."""
    cores_lista = [cores.get(k, '#999999') for k in serie.index] if cores else None
    fig = go.Figure(data=[go.Pie(
        labels=serie.index,
        values=serie.values,
        hole=0.4,
        marker=dict(colors=cores_lista),
        textinfo='label+value+percent',
        textfont=dict(size=13),
    )])
    fig.update_layout(
        title=dict(text=titulo, x=0.5, xanchor='center', font=dict(size=14)),
        showlegend=False,
        margin=dict(t=60, b=40, l=40, r=40),
        height=420,
    )
    fig.show()
    return fig

#### Reajuste de voto geral 2020 - 2025 (PV e PP)

In [15]:
# GERAL — PERÍODO — PV e PP (pizza com/sem reajuste)

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = df_final[df_final['ambiente'] == ambiente]
    serie = sub['teve_reajuste'].map({
        True:  'Com reajuste de voto',
        False: 'Sem reajuste de voto'
    }).value_counts()

    print(f"{ambiente}: {sub['teve_reajuste'].sum():,} com reajuste "
          f"de {len(sub):,} inclusões ({100*sub['teve_reajuste'].mean():.1f}%)")

    _ = plotar_pizza_reajuste(
        serie,
        titulo=f'Inclusões com reajuste de voto — {ambiente} (2020–2025)',
        cores=CORES_REAJUSTE,
    )

Plenário Virtual: 81 com reajuste de 4,807 inclusões (1.7%)


Plenário Físico: 77 com reajuste de 2,710 inclusões (2.8%)


#### Reajuste de voto geral 2020 - 2025 por ano (PV e PP)


In [16]:
# GRÁFICO 3 e 4 — GERAL — ANUAL — PV e PP (barras por ano)

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = df_final[(df_final['ambiente'] == ambiente) & (df_final['teve_reajuste'])]
    tab = sub.groupby('ano').size().reset_index(name='n')

    # Garante todos os anos 2020-2025, mesmo com zero
    tab = tab.set_index('ano').reindex(range(2020, 2026), fill_value=0).reset_index()

    _ = plotar_barras_reajuste(
        tab, col_x='ano',
        titulo=f'Inclusões com reajuste de voto por ano — {ambiente} (2020–2025)',
    )

#### Reajustes de votos geral 2020 - 2025 por ano e classe (PV e PP)

In [17]:
# GRÁFICO 5 e 6 — ANUAL — POR CLASSE — PV e PP (barras agrupadas por classe)

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = df_final[(df_final['ambiente'] == ambiente) & (df_final['teve_reajuste'])]
    tab = sub.groupby(['ano', 'classe']).size().reset_index(name='n')

    _ = plotar_barras_reajuste(
        tab, col_x='ano', col_grupo='classe',
        titulo=f'Reajuste de voto por ano e classe — {ambiente} (2020–2025)',
        cores=CORES_CLASSE,
    )

/tmp/ipykernel_751/209983182.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipykernel_751/209983182.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## Tramitações

O que significa "tramitou nos dois ambientes"
- Um processo (identificado pelo incidente) tramitou nos dois ambientes se ele tem pelo menos uma inclusão em pauta no Plenário Virtual E pelo menos uma no Plenário Físico. A unidade aqui é o processo, não a inclusão — porque estamos perguntando sobre a trajetória do processo.

### Tramitação nos dois ambientes

In [ ]:
df_final = tramitacao_dois_ambientes(df_final)

ambientes_por_processo = (
    df_final.groupby('incidente')
    .agg(tramitacao=('tramitacao', 'first'), classe=('classe', 'first'), tipo_questao=('tipo_questao', 'first'))
    .reset_index()
)

print("="*62)
print("TRAMITAÇÃO POR AMBIENTE (2020–2025)")
print("="*62)
print(f"Total de processos distintos:    {len(ambientes_por_processo):,}")

print("\nDistribuição por fluxo:")
contagem_tram = ambientes_por_processo['tramitacao'].value_counts()
total_proc = len(ambientes_por_processo)
max_tram = contagem_tram.max()

for fluxo, qtd in contagem_tram.items():
    pct = (qtd / total_proc) * 100
    barra = imprimir_barra_progresso(qtd, max_tram)
    print(f"  {fluxo:<20} {qtd:>5} ({pct:>5.1f}%) {barra}")

print("\nPor classe (Processos em Ambos os Ambientes):")
ambos = ambientes_por_processo[ambientes_por_processo['tramitacao'] == 'Ambos os ambientes']
contagem_classe = ambos['classe'].value_counts()
total_ambos = len(ambos)

for classe, qtd in contagem_classe.items():
    pct = (qtd / total_ambos) * 100
    print(f"  {classe:<6} {qtd:>4} ({pct:>5.1f}%)")

print("\n" + "="*62)


### Visualização

#### Tramitação nos dois ambientes geral 2020 - 2025

In [19]:
# GRÁFICO 1 — Tramitação nos dois ambientes — Geral — Período (pizza)
serie_tram = ambientes_por_processo['tramitacao'].value_counts()

CORES_TRAM = {
    'Ambos os ambientes': '#8b5cf6',   # roxo (o destaque)
    'Só Virtual':         '#2563eb',   # azul
    'Só Físico':          '#f59e0b',   # laranja
}

_ = plotar_pizza_reajuste(
    serie_tram,
    titulo='Tramitação por ambiente — processos CC (2020–2025)',
    cores=CORES_TRAM,
)

# Destaca o número de "ambos"
n_ambos = (ambientes_por_processo['tramitacao'] == 'Ambos os ambientes').sum()
print(f"\nProcessos que tramitaram nos DOIS ambientes: {n_ambos:,} "
      f"({100*n_ambos/len(ambientes_por_processo):.1f}%)")


Processos que tramitaram nos DOIS ambientes: 478 (16.9%)


#### Tramitação nos dois ambientes geral 2020 - 2025 por tipo de questão

In [34]:
# ===========================================================================
# GRÁFICO — Processos por tipo de tramitação — PERÍODO 2020–2025
# ===========================================================================
# Sem quebra por ano. Cada processo aparece UMA única vez, classificado
# pelo(s) ambiente(s) em que tramitou ao longo de todo o período.
# As barras somam exatamente 2.834 (total de processos distintos).

CORES_TRAMITACAO = {
    'Só Virtual':          '#2563eb',
    'Só Físico':           '#f59e0b',
    'Ambos os ambientes':  '#16a34a',
}

def classificar_tramitacao(ambientes):
    tem_v = 'Plenário Virtual' in ambientes
    tem_f = 'Plenário Físico' in ambientes
    if tem_v and tem_f:
        return 'Ambos os ambientes'
    return 'Só Virtual' if tem_v else 'Só Físico'

proc = (
    df_final.groupby('incidente')['ambiente']
    .apply(set)
    .reset_index(name='ambientes')
)
proc['tramitacao'] = proc['ambientes'].apply(classificar_tramitacao)

# Eixo X com valor único — mantém a assinatura da plotar_barras_stf
proc['periodo'] = '2020–2025'

# Conferência
print("Processos por tipo de tramitação (2020–2025):")
print(proc['tramitacao'].value_counts().to_string())
print(f"  Total: {len(proc):,}")

tab = proc.groupby(['periodo', 'tramitacao']).size().reset_index(name='n')
total = proc.groupby('periodo').size().reset_index(name='n')

print(f"\nSoma de todas as barras: {tab['n'].sum():,}  (deve ser 2.834)")

plotar_barras_stf(
    df_dados=tab, col_x='periodo', col_y='n', col_grupo='tramitacao',
    titulo='Processos por tipo de tramitação — 2020–2025',
    label_y='Processos (incidentes distintos)',
    mostrar_linha_total=True, df_total=total,
    cores=CORES_TRAMITACAO,
)

Processos por tipo de tramitação (2020–2025):
tramitacao
Só Virtual            2197
Ambos os ambientes     478
Só Físico              159
  Total: 2,834

Soma de todas as barras: 2,834  (deve ser 2.834)


NameError: name 'plotar_barras_stf' is not defined

In [21]:
# ===========================================================================
# GRÁFICO — Processos por tipo de tramitação — PERÍODO 2020–2025
# ===========================================================================
# Sem quebra por ano. Cada processo aparece uma única vez, classificado
# pelo(s) ambiente(s) em que tramitou ao longo de todo o período.
# Total: 2.834 processos distintos.

CORES_TRAMITACAO = {
    'Só Virtual':          '#2563eb',
    'Só Físico':           '#f59e0b',
    'Ambos os ambientes':  '#16a34a',
}

def classificar_tramitacao(ambientes):
    tem_v = 'Plenário Virtual' in ambientes
    tem_f = 'Plenário Físico' in ambientes
    if tem_v and tem_f:
        return 'Ambos os ambientes'
    return 'Só Virtual' if tem_v else 'Só Físico'

proc = (
    df_final.groupby('incidente')['ambiente']
    .apply(set)
    .reset_index(name='ambientes')
)
proc['tramitacao'] = proc['ambientes'].apply(classificar_tramitacao)

serie = proc['tramitacao'].value_counts()

print(f"Processos por tipo de tramitação (2020–2025):")
for cat, n in serie.items():
    print(f"  {cat:<22} {n:>5,} ({100*n/len(proc):>5.1f}%)")
print(f"  {'─'*38}")
print(f"  {'Total':<22} {len(proc):>5,}")

_ = plotar_pizza_reajuste(
    serie,
    titulo='Processos por tipo de tramitação — 2020–2025',
    cores=CORES_TRAMITACAO,
)

Processos por tipo de tramitação (2020–2025):
  Só Virtual             2,197 ( 77.5%)
  Ambos os ambientes       478 ( 16.9%)
  Só Físico                159 (  5.6%)
  ──────────────────────────────────────
  Total                  2,834


In [33]:
# ===========================================================================
# GRÁFICO — Processos por tipo de tramitação — PERÍODO 2020–2025
# ===========================================================================
# Sem quebra por ano. Cada processo aparece uma única vez, classificado
# pelo(s) ambiente(s) em que tramitou ao longo de todo o período.
# As barras somam exatamente 2.834 (total de processos distintos).

CORES_TRAMITACAO = {
    'Só Virtual':          '#2563eb',
    'Só Físico':           '#f59e0b',
    'Ambos os ambientes':  '#16a34a',
}

def classificar_tramitacao(ambientes):
    tem_v = 'Plenário Virtual' in ambientes
    tem_f = 'Plenário Físico' in ambientes
    if tem_v and tem_f:
        return 'Ambos os ambientes'
    return 'Só Virtual' if tem_v else 'Só Físico'

proc = (
    df_final.groupby('incidente')['ambiente']
    .apply(set)
    .reset_index(name='ambientes')
)
proc['tramitacao'] = proc['ambientes'].apply(classificar_tramitacao)

# Ordena as categorias de forma fixa (não pela contagem)
ORDEM = ['Só Virtual', 'Só Físico', 'Ambos os ambientes']
tab = (
    proc['tramitacao'].value_counts()
    .reindex(ORDEM, fill_value=0)
    .reset_index()
)
tab.columns = ['tramitacao', 'n']

print(f"Processos por tipo de tramitação (2020–2025):")
for _, r in tab.iterrows():
    print(f"  {r['tramitacao']:<22} {r['n']:>5,} ({100*r['n']/len(proc):>5.1f}%)")
print(f"  {'─'*38}")
print(f"  {'Total':<22} {len(proc):>5,}")

_ = plotar_barras_reajuste(
    tab, col_x='tramitacao', col_grupo=None,
    titulo='Processos por tipo de tramitação — 2020–2025',
    label_y='Processos (incidentes distintos)',
    cores=CORES_TRAMITACAO,
)

Processos por tipo de tramitação (2020–2025):
  Só Virtual             2,197 ( 77.5%)
  Só Físico                159 (  5.6%)
  Ambos os ambientes       478 ( 16.9%)
  ──────────────────────────────────────
  Total                  2,834


### Salvamento

In [ ]:
print("="*62)
print("CONSOLIDAÇÃO DE TRAMITAÇÃO")
print("="*62)

print("Distribuição (por inclusão total):")
contagem_inc = df_final['tramitacao'].value_counts()
total_inc = len(df_final)
max_inc = contagem_inc.max()

for fluxo, qtd in contagem_inc.items():
    pct = (qtd / total_inc) * 100
    barra = imprimir_barra_progresso(qtd, max_inc)
    print(f"  {fluxo:<20} {qtd:>5,} ({pct:>5.1f}%) {barra}")

print("\nDistribuição (por processos únicos):")
contagem_proc = ambientes_por_processo['tramitacao'].value_counts()
total_proc = len(ambientes_por_processo)
max_proc = contagem_proc.max()

for fluxo, qtd in contagem_proc.items():
    pct = (qtd / total_proc) * 100
    barra = imprimir_barra_progresso(qtd, max_proc)
    print(f"  {fluxo:<20} {qtd:>5,} ({pct:>5.1f}%) {barra}")

# ── Salvamento ───────────────────────────────────────────────────────────
df_final.to_parquet(PROCESSED_PATH / 'tramitacoes.parquet', index=False)

print("\n" + "="*62)
print(f"✓ Dataset salvo: {PROCESSED_PATH / 'tramitacoes.parquet'}")
print("="*62)


## Sustentação oral

RC (recursos como AgR, ED) não têm direito a sustentação oral, e inclusões "retiradas de pauta" nunca chegaram a julgamento — então nenhuma dessas deveria estar no universo de análise. Incluí-las distorceria a taxa de sustentação (inflaria o denominador com casos onde sustentação era impossível ou não se aplicava).

Universo de análise = inclusões em pauta, EXCETO:
- As de tipo de questão RC (sem direito a sustentação)
as com desfecho "retirado de pauta" (não foram julgadas)
- Ou seja, o universo fica só com PR e IJ (QI) que não foram retiradas. E sobre esse universo, calculamos a taxa de sustentação, quebrada por tipo de questão (PR/QI) e por desfecho.

### Verificação da sustentação oral

In [ ]:
sust = verificar_sustentacao_oral(df_andamentos)
df_final = vincular_sustentacao_a_inclusoes(df_final, sust)

# ── Universo de Análise ──────────────────────────────────────────────────
universo_sust = df_final[(df_final['tipo_questao'] != 'RC') & (df_final['macro_desfecho'] == 'Concluído')].copy()
universo_sust['tipo_questao'] = universo_sust['tipo_questao'].replace({'IJ': 'QI'})

print("="*62)
print("UNIVERSO DE ANÁLISE — SUSTENTAÇÃO ORAL")
print("="*62)
print(f"Total de inclusões (df_final):        {len(df_final):,}")
print(f"Universo de análise final:            {len(universo_sust):,}")

print("\nTaxa por ambiente:")
for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = universo_sust[universo_sust['ambiente'] == ambiente]
    n = sub['teve_sustentacao'].sum()
    pct = (n / len(sub)) * 100 if len(sub) > 0 else 0
    barra = imprimir_barra_progresso(n, len(sub))
    print(f"  {ambiente:<18} {n:>4} de {len(sub):>4} ({pct:>5.1f}%) {barra}")

print("\n" + "="*62)
print("SUSTENTAÇÃO POR TIPO DE QUESTÃO")
print("="*62)
for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = universo_sust[universo_sust['ambiente'] == ambiente]
    if not sub.empty:
        print(f"{ambiente}:")
        for tipo in ['PR', 'QI']:
            sub_t = sub[sub['tipo_questao'] == tipo]
            if not sub_t.empty:
                n = sub_t['teve_sustentacao'].sum()
                pct = (n / len(sub_t)) * 100
                print(f"  {tipo}: {n:>4} de {len(sub_t):>4} ({pct:>5.1f}%)")

print("\n" + "="*62)
print("SUSTENTAÇÃO POR TIPO DE DESFECHO")
print("="*62)
for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = universo_sust[universo_sust['ambiente'] == ambiente]
    if not sub.empty:
        print(f"{ambiente}:")
        tab = sub.groupby('desfecho')['teve_sustentacao'].agg(['sum', 'count']).reset_index()
        tab['pct'] = (100 * tab['sum'] / tab['count']).round(1)
        for _, r in tab.sort_values('sum', ascending=False).iterrows():
            print(f"  {r['desfecho']:<45} {int(r['sum']):>4} ({r['pct']:>5.1f}%)")
print("="*62)


In [24]:
# Controle de qualidade — RCs que tinham sustentação (agora excluídos)
rc_com_sust = df_final[
    (df_final['tipo_questao'] == 'RC') & (df_final['teve_sustentacao'])
]
print(f"RCs com sustentação detectada (excluídos do universo): {len(rc_com_sust):,}")
print(f"  ({100*len(rc_com_sust)/(df_final['tipo_questao']=='RC').sum():.1f}% dos RCs)")

RCs com sustentação detectada (excluídos do universo): 30
  (2.7% dos RCs)


### Visualização

In [25]:
CORES_SUST = {
    'Com sustentação oral': '#0891b2',   # ciano
    'Sem sustentação oral': '#e5e7eb',   # cinza claro
}

#### Sustentação oral geral 2020 - 2025 (PV e PP)

In [26]:
# ---------------------------------------------------------------------------
# GRÁFICO 1 e 2 — SUSTENTAÇÃO ORAL — PERÍODO — PV e PP (pizza)
# ---------------------------------------------------------------------------
# Universo: apenas inclusões CONCLUÍDAS, excluindo tipo de questão RC
# (recursos não têm direito a sustentação oral)

CORES_SUST = {
    'Com sustentação oral': '#0891b2',   # ciano
    'Sem sustentação oral': '#e5e7eb',   # cinza claro
}

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = universo_sust[universo_sust['ambiente'] == ambiente]

    if len(sub) == 0:
        print(f"{ambiente}: sem dados no universo de análise")
        continue

    serie = sub['teve_sustentacao'].map({
        True:  'Com sustentação oral',
        False: 'Sem sustentação oral'
    }).value_counts()

    print(f"{ambiente}: {sub['teve_sustentacao'].sum():,} com sustentação "
          f"de {len(sub):,} concluídas ({100*sub['teve_sustentacao'].mean():.1f}%)")

    _ = plotar_pizza_reajuste(
        serie,
        titulo=f'Sustentação oral em inclusões concluídas — {ambiente} (2020–2025)',
        cores=CORES_SUST,
    )

Plenário Virtual: 621 com sustentação de 2,128 concluídas (29.2%)


Plenário Físico: 60 com sustentação de 276 concluídas (21.7%)


#### Sustentação oral por tipo de desfecho (PV e PP)

In [27]:
# ===========================================================================
# SUSTENTAÇÃO ORAL POR TIPO DE DESFECHO — PIZZA — PV e PP
# ===========================================================================
# Mostra, entre as inclusões que tiveram sustentação oral, como elas se
# distribuem pelos tipos de desfecho.

CORES_DESFECHO = {
    'Concluído - decisão unânime':                    '#16a34a',
    'Concluído - decisão maioria com o relator':      '#2563eb',
    'Concluído - decisão maioria, vencido o relator': '#f59e0b',
}

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = universo_sust[
        (universo_sust['ambiente'] == ambiente) & (universo_sust['teve_sustentacao'])
    ]
    if len(sub) == 0:
        print(f"{ambiente}: sem casos com sustentação no universo")
        continue

    serie = sub['desfecho'].value_counts()

    print(f"{ambiente}: {len(sub):,} inclusões com sustentação oral")

    _ = plotar_pizza_reajuste(
        serie,
        titulo=f'Desfecho das inclusões com sustentação oral — {ambiente} (2020–2025)',
        cores=CORES_DESFECHO,
    )

Plenário Virtual: 621 inclusões com sustentação oral


Plenário Físico: 60 inclusões com sustentação oral


#### Desfecho das inclusões sem sustentação oral (PV e PP)

In [28]:
# ===========================================================================
# DESFECHO DAS INCLUSÕES SEM SUSTENTAÇÃO ORAL — PIZZA — PV e PP
# ===========================================================================
# Mostra, entre as inclusões que NÃO tiveram sustentação oral, como elas se
# distribuem pelos tipos de desfecho.

CORES_DESFECHO = {
    'Concluído - decisão unânime':                    '#16a34a',
    'Concluído - decisão maioria com o relator':      '#2563eb',
    'Concluído - decisão maioria, vencido o relator': '#f59e0b',
}

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = universo_sust[
        (universo_sust['ambiente'] == ambiente) & (~universo_sust['teve_sustentacao'])
    ]
    if len(sub) == 0:
        print(f"{ambiente}: sem casos sem sustentação no universo")
        continue

    serie = sub['desfecho'].value_counts()

    print(f"{ambiente}: {len(sub):,} inclusões sem sustentação oral")

    _ = plotar_pizza_reajuste(
        serie,
        titulo=f'Desfecho das inclusões sem sustentação oral — {ambiente} (2020–2025)',
        cores=CORES_DESFECHO,
    )

Plenário Virtual: 1,507 inclusões sem sustentação oral


Plenário Físico: 216 inclusões sem sustentação oral


In [29]:
# ===========================================================================
# ALTERNATIVA — uma pizza por desfecho (com/sem sustentação)
# ===========================================================================
CORES_SUST = {
    'Com sustentação oral': '#0891b2',
    'Sem sustentação oral': '#e5e7eb',
}

DESFECHOS = [
    'Concluído - decisão unânime',
    'Concluído - decisão maioria com o relator',
    'Concluído - decisão maioria, vencido o relator',
]

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    for desf in DESFECHOS:
        sub = universo_sust[
            (universo_sust['ambiente'] == ambiente) &
            (universo_sust['desfecho'] == desf)
        ]
        if len(sub) == 0:
            continue

        serie = sub['teve_sustentacao'].map({
            True:  'Com sustentação oral',
            False: 'Sem sustentação oral'
        }).value_counts()

        _ = plotar_pizza_reajuste(
            serie,
            titulo=f'{desf} — {ambiente} (2020–2025)',
            cores=CORES_SUST,
        )

#### Sustentação oral geral 2020 - 2025 por ano (PV e PP)

In [30]:
# ---------------------------------------------------------------------------
# GRÁFICO 3 e 4 — ANUAL — PV e PP (barras por ano)
# ---------------------------------------------------------------------------
for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    sub = df_final[(df_final['ambiente'] == ambiente) & (df_final['teve_sustentacao'])]
    tab = sub.groupby('ano').size().reset_index(name='n')
    tab = tab.set_index('ano').reindex(range(2020, 2026), fill_value=0).reset_index()

    _ = plotar_barras_reajuste(
        tab, col_x='ano',
        titulo=f'Inclusões com sustentação oral por ano — {ambiente} (2020–2025)',
    )

### Salvamento

In [31]:
# ===========================================================================
# CONSOLIDAÇÃO — salva o dataset mestre com todas as marcações
# ===========================================================================

# Confirma que todas as colunas derivadas existem
colunas_derivadas = ['teve_reajuste', 'tramitacao', 'tramitou_ambos', 'teve_sustentacao']
faltando = [c for c in colunas_derivadas if c not in df_final.columns]

if faltando:
    print(f"⚠ Colunas ausentes (rode as células correspondentes antes): {faltando}")
else:
    print("Todas as marcações presentes no df_final.")

    # Resumo geral
    print(f"\n{'='*55}")
    print(f"RESUMO DAS MARCAÇÕES (por inclusão em pauta)")
    print(f"{'='*55}")
    print(f"Total de inclusões: {len(df_final):,}")
    print(f"  Com reajuste de voto:  {df_final['teve_reajuste'].sum():,} ({100*df_final['teve_reajuste'].mean():.1f}%)")
    print(f"  Com sustentação oral:  {df_final['teve_sustentacao'].sum():,} ({100*df_final['teve_sustentacao'].mean():.1f}%)")
    print(f"  Tramitou em ambos:     {df_final['tramitou_ambos'].sum():,} ({100*df_final['tramitou_ambos'].mean():.1f}%)")

    # Salva
    df_final.to_parquet(
        PROCESSED_PATH / 'sustentacao_oral.parquet', index=False
    )
    print(f"\nDataset mestre salvo: sustentacao_oral.parquet")
    print(f"Colunas: {df_final.columns.tolist()}")

Todas as marcações presentes no df_final.

RESUMO DAS MARCAÇÕES (por inclusão em pauta)
Total de inclusões: 7,517
  Com reajuste de voto:  158 (2.1%)
  Com sustentação oral:  1,913 (25.4%)
  Tramitou em ambos:     3,072 (40.9%)

Dataset mestre salvo: sustentacao_oral.parquet
Colunas: ['incidente', 'nome_processo', 'classe', 'relator', 'ano', 'data_inclusao', 'data_inclusao_dt', 'ambiente', 'tipo_questao', 'tipo_questao_original', 'sufixo_extraido', 'desfecho', 'macro_desfecho', 'andamento_origem', 'virou_sessao', 'teve_reajuste', 'tramitacao', 'tramitou_ambos', 'teve_sustentacao']


## Salvamento

In [32]:
df_final.to_parquet(
    PROCESSED_PATH / 'inclusoes_em_pauta.parquet', index=False
)